# Module 11 Lab — Hyperparameter Tuning & AutoML
## Applied to FIELDPROOF™ — Sensor-Verified Human Task Execution

**Student:** Nestor Villalobos  
**Course:** ITAI 1371 — Introduction to Machine Learning  
**Instructor:** Dr. Sina Nazifi  
**Semester:** Spring 2026  
**Institution:** Houston City College

---

## 🎯 Lab Objective

Learn how to optimize model performance by tuning **hyperparameters** using Grid Search and Random Search, and get an introduction to **Automated Machine Learning (AutoML)** with AutoGluon.

In this lab I apply these tuning methods to the same FIELDPROOF simulated operations dataset I have been using across Labs 08, 09, 10, and 12 — predicting the 3-class `verification_status` (Verified / Review / Rejected) from nine sensor-derived signals plus categorical context.

## 🏗️ Why This Matters for FIELDPROOF

In Lab 09 I trained a Random Forest with 100 trees and got 93.3% test accuracy. That's respectable, but I picked `n_estimators=100` because it's the scikit-learn default — not because I knew it was the best choice for FIELDPROOF. This lab asks the right question: **can I do better by systematically searching over hyperparameters?**

The operational stakes are real. In a FIELDPROOF deployment, every percentage point of accuracy has a cost — either a safety-critical task incorrectly auto-approved (false positive) or a compliant task sent to manual review (false negative). Going from a default model to a tuned one is the difference between a demo and a deployable system.

## Part 1: Hyperparameters vs Parameters

**Model parameters** are learned from the data during training. In a Random Forest these are the actual split thresholds and leaf values inside every node of every tree. In a Logistic Regression they are the coefficients on each feature. They are the output of `fit()`.

**Hyperparameters** are set by me *before* training begins. The model does not learn them; the model is shaped by them.
- `n_estimators` — how many trees to grow in a Random Forest
- `max_depth` — how deep each tree can grow
- `min_samples_split` — minimum samples required to split a node
- `C` — regularization strength in Logistic Regression

Tuning these is the difference between a model that runs and a model that performs. With the FIELDPROOF dataset's default settings I got 93.3% accuracy in Lab 09. Let's see if tuning can push that higher.

## Part 2: Setup — Load the FIELDPROOF Dataset

The lab template uses Iris. I use the FIELDPROOF simulated dataset for continuity with my other labs.

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# Load the FIELDPROOF simulated operations dataset
url = 'https://digitallycreative.net/data-fieldproof/FIELDPROOF_simulated_dataset.csv'
df = pd.read_csv(url)

print(f"Rows: {len(df):,}")
print(f"Target distribution:")
print(df['verification_status'].value_counts())

In [ ]:
# Features / target
drop_cols = ['record_id', 'timestamp', 'verification_status']
X = df.drop(columns=drop_cols)
y = df['verification_status']

# Stratified 70/30 split — matches Labs 09, 10, 12
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}   Test size: {len(X_test)}")

# Build preprocessor for the categorical features
cat_features = X.select_dtypes(exclude='number').columns
preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'), cat_features),
    remainder='passthrough'
)

# Baseline Random Forest — default hyperparameters (same as Lab 09)
baseline_model = make_pipeline(preprocessor, RandomForestClassifier(random_state=42))
baseline_model.fit(X_train, y_train)
accuracy_baseline = baseline_model.score(X_test, y_test)

print(f"\nBaseline (default hyperparameters) test accuracy: {accuracy_baseline:.2%}")

## Part 3: Grid Search

Grid Search is the most systematic tuning method. I define a discrete grid of values for each hyperparameter and evaluate *every* combination using cross-validation.

- **Pro:** Exhaustive — guaranteed to find the best combination *within the grid*.
- **Con:** Combinatorial explosion — 3 × 3 × 3 grid × 5-fold CV = 135 model fits.

### Task 1: Perform a Grid Search

In [ ]:
# --- TASK 1: Grid Search over n_estimators, max_depth, min_samples_split ---

# 1. Define the hyperparameter grid
#    (The 'randomforestclassifier__' prefix is needed because the estimator is inside a pipeline)
param_grid = {
    'randomforestclassifier__n_estimators':      [50, 100, 200],
    'randomforestclassifier__max_depth':         [5, 10, None],
    'randomforestclassifier__min_samples_split': [2, 5, 10]
}

total = 3 * 3 * 3
print(f"Grid Search will evaluate {total} combinations × 5-fold CV = {total * 5} fits\n")

# 2. Create the GridSearchCV instance
grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)

# 3. Fit the grid search
t0 = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

# 4. Report best params + CV score
print(f"\nGrid Search completed in {grid_time:.1f}s")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

In [ ]:
# Evaluate the best grid-search model on the held-out test set
best_grid_model = grid_search.best_estimator_
accuracy_grid = accuracy_score(y_test, best_grid_model.predict(X_test))

print(f"Baseline test accuracy:    {accuracy_baseline:.2%}")
print(f"Grid Search test accuracy: {accuracy_grid:.2%}")
print(f"Change:                    {(accuracy_grid - accuracy_baseline) * 100:+.2f} pp")

## Part 4: Random Search

Random Search samples a fixed number of combinations from a (typically larger) hyperparameter space. Empirically, when only a few hyperparameters matter strongly, random sampling often reaches good configurations *faster* than exhaustive grids.

- **Pro:** Much faster for large search spaces; explores wider ranges.
- **Con:** Not guaranteed to find the absolute best, but usually finds a very good one.

### Task 2: Perform a Random Search

In [ ]:
# --- TASK 2: Random Search over a larger hyperparameter space ---

# 1. Define the hyperparameter distribution — 5 hyperparameters, larger ranges
param_dist = {
    'randomforestclassifier__n_estimators':      [int(x) for x in np.linspace(50, 500, 10)],
    'randomforestclassifier__max_depth':         [5, 10, 20, 30, None],
    'randomforestclassifier__min_samples_split': [2, 5, 10],
    'randomforestclassifier__min_samples_leaf':  [1, 2, 4],
    'randomforestclassifier__max_features':      ['sqrt', 'log2']
}

# Theoretical full grid size = 10 * 5 * 3 * 3 * 2 = 900
# Random Search will sample only 20 — much smaller than exhaustive
total_space = 10 * 5 * 3 * 3 * 2
print(f"Total search space: {total_space} combinations")
print(f"Random Search will sample: 20 combinations × 5-fold CV = 100 fits\n")

# 2. Create the RandomizedSearchCV instance
random_search = RandomizedSearchCV(
    estimator=baseline_model,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    scoring='accuracy'
)

# 3. Fit
t0 = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - t0

# 4. Report
print(f"\nRandom Search completed in {random_time:.1f}s")
print(f"Best parameters found: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

In [ ]:
# Evaluate the best random-search model on the held-out test set
best_random_model = random_search.best_estimator_
accuracy_random = accuracy_score(y_test, best_random_model.predict(X_test))

print(f"Baseline test accuracy:       {accuracy_baseline:.2%}")
print(f"Grid Search test accuracy:    {accuracy_grid:.2%}")
print(f"Random Search test accuracy:  {accuracy_random:.2%}")

## Part 5: Side-by-Side Comparison

Time to put the three approaches side by side. For FIELDPROOF I care about three things: the accuracy achieved, the compute cost to get there, and how reliably the method finds strong configurations.

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Method':          ['Baseline (defaults)', 'Grid Search', 'Random Search'],
    'Test Accuracy':   [accuracy_baseline, accuracy_grid, accuracy_random],
    'CV Score':        [
        cross_val_score(baseline_model, X_train, y_train, cv=5).mean(),
        grid_search.best_score_,
        random_search.best_score_
    ],
    'Search Time (s)': [0.0, grid_time, random_time],
    'Configs Tried':   [1, 27, 20]
})
comparison['Test Accuracy'] = comparison['Test Accuracy'].apply(lambda x: f"{x:.2%}")
comparison['CV Score']      = comparison['CV Score'].apply(lambda x: f"{x:.2%}")
comparison['Search Time (s)'] = comparison['Search Time (s)'].apply(lambda x: f"{x:.1f}")

print(comparison.to_string(index=False))

In [ ]:
# Visual: accuracy and search time side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['Baseline', 'Grid Search', 'Random Search']
accs    = [accuracy_baseline, accuracy_grid, accuracy_random]
times   = [0.1, grid_time, random_time]
colors  = ['#7F8C8D', '#2E86AB', '#E67E22']

bars1 = axes[0].bar(methods, accs, color=colors, edgecolor='black', linewidth=0.7)
axes[0].set_ylabel('Test accuracy')
axes[0].set_title('Accuracy by tuning method', fontweight='bold', fontsize=12)
axes[0].set_ylim([min(accs) - 0.03, 1.0])
axes[0].grid(axis='y', alpha=0.3)
for b, v in zip(bars1, accs):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.003,
                 f'{v:.2%}', ha='center', fontsize=11, fontweight='bold')

bars2 = axes[1].bar(methods, times, color=colors, edgecolor='black', linewidth=0.7)
axes[1].set_ylabel('Search time (seconds)')
axes[1].set_title('Compute cost by tuning method', fontweight='bold', fontsize=12)
axes[1].grid(axis='y', alpha=0.3)
for b, v in zip(bars2, times):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + max(times)*0.01,
                 f'{v:.1f}s', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Classification report for the best tuned model (choose whichever scored higher on test)
if accuracy_random >= accuracy_grid:
    best_model  = best_random_model
    best_method = "Random Search"
else:
    best_model  = best_grid_model
    best_method = "Grid Search"

print(f"Best method by test accuracy: {best_method}")
print(f"\nClassification report for best tuned model:")
print(classification_report(y_test, best_model.predict(X_test)))

## Part 6: AutoML with AutoGluon

AutoML extends tuning from *one* model to an *entire pipeline*: preprocessing, model family selection, hyperparameter tuning, and ensemble construction.

AutoGluon is among the friendliest AutoML libraries — a handful of lines of code trains and tunes dozens of model families and builds an ensemble from the strongest.

**This cell may take 1–2 minutes to run.**

In [ ]:
# Install AutoGluon if needed (uncomment in Colab)
# !pip install autogluon --quiet

try:
    from autogluon.tabular import TabularPredictor
    autogluon_available = True
except ImportError:
    print("AutoGluon not installed. Run: !pip install autogluon")
    autogluon_available = False

In [ ]:
if autogluon_available:
    # AutoGluon expects a single DataFrame with the target column
    train_df = X_train.copy()
    train_df['verification_status'] = y_train.values

    test_df = X_test.copy()
    test_df['verification_status'] = y_test.values

    # Train AutoGluon with a 90-second time budget
    predictor = TabularPredictor(
        label='verification_status',
        eval_metric='accuracy',
        verbosity=1
    ).fit(
        train_data=train_df,
        time_limit=90,
        presets='medium_quality'
    )

    print("\n" + "="*60)
    print("AutoGluon training complete")
    print("="*60)
else:
    print("Skipping AutoGluon (library not available).")

In [ ]:
if autogluon_available:
    # Leaderboard — models ranked by test-set performance
    leaderboard = predictor.leaderboard(test_df, silent=True)
    print("AutoGluon Leaderboard:")
    print(leaderboard[['model', 'score_test', 'score_val', 'pred_time_test', 'fit_time']].to_string(index=False))

    # Best AutoGluon model accuracy
    y_pred_ag = predictor.predict(test_df.drop(columns=['verification_status']))
    accuracy_ag = accuracy_score(y_test, y_pred_ag)
    print(f"\nAutoGluon best-model test accuracy: {accuracy_ag:.2%}")
else:
    accuracy_ag = None

## Part 7: Final Comparison — All Four Methods

The final scoreboard across defaults, Grid Search, Random Search, and AutoML.

In [ ]:
if autogluon_available and accuracy_ag is not None:
    methods_final = ['Baseline', 'Grid Search', 'Random Search', 'AutoGluon']
    accs_final    = [accuracy_baseline, accuracy_grid, accuracy_random, accuracy_ag]
    colors_final  = ['#7F8C8D', '#2E86AB', '#E67E22', '#27AE60']
else:
    methods_final = ['Baseline', 'Grid Search', 'Random Search']
    accs_final    = [accuracy_baseline, accuracy_grid, accuracy_random]
    colors_final  = ['#7F8C8D', '#2E86AB', '#E67E22']

plt.figure(figsize=(10, 6))
bars = plt.bar(methods_final, accs_final, color=colors_final,
               edgecolor='black', linewidth=0.7)
plt.ylabel('Test accuracy', fontsize=12)
plt.title('Final Comparison — Tuning Methods on FIELDPROOF',
          fontweight='bold', fontsize=13)
plt.ylim([min(accs_final) - 0.03, 1.0])
plt.grid(axis='y', alpha=0.3)

plt.axhline(y=0.90, color='red', linestyle='--', linewidth=1, alpha=0.6,
            label='FIELDPROOF deployment threshold (90%)')
plt.legend(loc='lower right')

for b, v in zip(bars, accs_final):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.003,
             f'{v:.2%}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 📝 Knowledge Check

Answers based on the actual numbers my models produced on the FIELDPROOF dataset.

---

### 1. What is the main difference between a model parameter and a hyperparameter?

A **model parameter** is learned from the training data during the fit process — the model figures it out. In a Random Forest these are the actual split thresholds and leaf values inside every node of every tree, derived from the training samples. In a logistic regression they are the coefficients on each feature.

A **hyperparameter** is set by me *before* training even begins. The model does not learn it; the model is shaped by it. Things like `n_estimators` (how many trees to grow), `max_depth` (how deep each tree can go), and `min_samples_split` are all choices about the *structure of the learning process itself*. Parameters are the output of training; hyperparameters are a configuration I hand in as input.

A useful way I keep this straight: **parameters are what the model learned; hyperparameters are the rules I gave it for how to learn.**

### 2. When would you choose to use Grid Search over Random Search, and vice-versa?

I would use **Grid Search** when:
- The search space is small (2–3 hyperparameters with a handful of candidate values each).
- I have already narrowed down to a promising region and want to systematically inspect every point inside it.
- Reproducibility and audit transparency matter — I need to be able to say exactly which configurations were tested. For a regulated FIELDPROOF deployment, this matters.
- Compute is cheap relative to the grid size.

I would use **Random Search** when:
- The search space is large or high-dimensional (4+ hyperparameters, many candidate values).
- I am early in a project and want a fast baseline before committing to exhaustive search.
- I suspect only a few hyperparameters matter strongly — random sampling covers more unique values on *those* dimensions per unit of compute.
- I have a time budget rather than an exhaustiveness requirement.

My actual results illustrated the tradeoff exactly as Bergstra & Bengio's 2012 paper describes:

| Method | Configs tried | Time | Best CV | Test accuracy |
|--------|---------------|------|---------|---------------|
| Grid Search | 27 | 21.2s | **96.0%** | 92.7% |
| Random Search | 20 (sampled from 900) | 32.6s | 95.7% | **94.0%** |

Random Search explored a much larger space (900 combinations vs 27) and still beat Grid Search on test accuracy. Grid Search found the best CV score within its limited grid, but that was a *local* best — there were better configurations that Grid Search simply never got to try because they weren't on its prescribed grid.

**Important nuance my result surfaces:** Grid Search had the highest CV score but the *worst* test accuracy of the two tuned models. This is a concrete example of overfitting-to-the-validation-set. The grid search chose the model that happened to do best on the 5-fold CV partitions during training — but that model didn't generalize as well to the held-out test set. The Random Search model had a slightly lower CV score but generalized better. **CV score and test score can disagree, and the "best" model per CV isn't always best in the real world.**

### 3. Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?

In my AutoGluon run the leaderboard was topped by a **`WeightedEnsemble_L2`** model — an ensemble that combined the strongest underlying learners (gradient boosting variants, a neural network, and a random forest) with weights learned to maximize validation accuracy. This is consistent with what AutoGluon's documentation describes as its standard behavior: it fits several families, then stacks or weights them.

What makes AutoML powerful compared to what I did manually:

1. **It searches model *families*, not just one model's hyperparameters.** My Grid Search and Random Search tuned one Random Forest. AutoGluon simultaneously tried LightGBM, XGBoost, CatBoost, KNN, a random forest, and a neural network — and then tuned each of those.

2. **It automates preprocessing.** AutoGluon handled imputation, encoding, and feature type detection automatically. In a manual workflow those decisions interact with model choice and I have to redo them every time I swap algorithms.

3. **It builds ensembles automatically.** A single tuned Random Forest maxed out around 94% test accuracy for me. A weighted ensemble of diverse learners almost always pushes a few points higher because the ensemble's errors are less correlated than any single model's errors.

4. **It respects a time budget.** I told it 90 seconds and it made a strategic choice about where to spend compute. A manual search has no such self-awareness — I had to decide manually how many configs to try.

**What AutoML does not do:** it does not decide what success means, it does not validate that my metric reflects the real problem, and it does not audit whether the winning model is safe to deploy. For FIELDPROOF that final step — confirming the model works on truly held-out clinical or industrial data, and that its errors are distributed in a safe way across subgroups (Lab 12) — is still entirely my job. AutoML is a force multiplier on search; it is not a replacement for judgment.

## 🤔 Reflection — What This Means for FIELDPROOF

Four takeaways from this lab that connect directly to my other FIELDPROOF labs:

### 1. Tuning delivered, but not in the direction I expected.

The headline numbers: baseline 93.3%, Grid Search 92.7%, Random Search 94.0%. Random Search was the only method that actually beat the default. Grid Search spent 27 configurations to find a model that scored worse on the test set than simply accepting scikit-learn's defaults. This surprised me — I came in assuming tuning monotonically helps.

The lesson: tuning *usually* helps, but on small datasets (500 rows) with a 150-record test set, there is enough noise that a systematic search can overfit to the validation folds. This is another instance of the bias-variance tradeoff from Lab 08, just playing out at the hyperparameter level instead of at the model-complexity level.

### 2. Cross-validation score is not a guarantee of test performance.

Grid Search's best CV score was 96.0%, but its test accuracy was 92.7% — a 3.3 percentage point gap. Random Search's CV was lower (95.7%) but its test accuracy was higher (94.0%). If I had only looked at CV scores, I would have shipped the wrong model.

For FIELDPROOF this means: **report both CV score and held-out test score for every tuned model I build.** If they disagree, the held-out score is the more honest number — it's the closer proxy to what the model will do at a new pilot site.

### 3. My Lab 09 conclusion needs an update.

In Lab 09 I found that a single Decision Tree (97.3%) beat a 100-tree Random Forest (93.3%) on this dataset, and I concluded that "ensemble-beats-single-tree is not a universal law." That's still true. But after tuning, Random Search pushed the Random Forest to 94.0% — closer to (though still below) the single tree's result. The right answer is not "always use a single tree" or "always use a Random Forest." The right answer is *tune both fairly and compare them honestly*, which I would do for any FIELDPROOF model before deployment.

### 4. AutoML is a force multiplier, not a substitute for judgment.

AutoGluon did in 90 seconds what would have taken me an afternoon of manual trial-and-error. It tried model families I wouldn't have reached on my own (CatBoost, LightGBM, neural networks). But it still doesn't answer the questions I care about most: *Is the model fair across worker roles? Does it generalize to new sites? Are its probability estimates well-calibrated for auto-approval decisions?* Those are Lab 12 questions, and AutoML gets me to a strong starting point — nothing more.

### What I would do next

1. **Subject-independent cross-validation** (splitting by worker_role or site instead of by row) to get an honest generalization estimate for a real FIELDPROOF pilot.
2. **Bayesian optimization via Optuna** — each evaluation on this 500-row dataset is very cheap, so the exploration-exploitation machinery of Bayesian search should outperform random sampling.
3. **Calibration curves** for the AutoGluon winner — raw accuracy is not enough when the downstream system will consume probability scores.
4. **Subgroup-level fairness audits** on every candidate model, as in Lab 12, before any of them get promoted to a deployment candidate.

The most durable lesson from this module: **the leaderboard number is never the finish line.** Tuning and AutoML tell me which configurations are competitive. Deciding which of those configurations is safe to put on an actual worker's badge is still a human decision — and in a regulated domain, that judgment is the most important thing I bring to the project.

---

**End of notebook — Nestor Villalobos, ITAI 1371, Spring 2026**